In [14]:
import os
import time
import json
import requests
import pandas as pd

from pathlib import Path
from difflib import SequenceMatcher
from dotenv import load_dotenv

In [ ]:
load_dotenv()

NYT_API_KEY = os.getenv("NYT_API_KEY")

START_YEAR = 2004
END_YEAR = 2024
SAMPLE_SIZE = 25

DATA_RAW = Path("data/raw")
DATA_PROCESSED = Path("data/processed")
DATA_CACHE = Path("data/cache/nyt")

DATA_RAW.mkdir(parents=True, exist_ok=True)
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
DATA_CACHE.mkdir(parents=True, exist_ok=True)

NYT_ARTICLE_SEARCH_URL = "https://api.nytimes.com/svc/search/v2/articlesearch.json"

if not NYT_API_KEY:
    raise ValueError("Missing NYT_API_KEY in .env")

In [ ]:
DATA_RAW = Path("data/raw")

basics_path = DATA_RAW / "title.basics.tsv"
akas_path = DATA_RAW / "title.akas.tsv"

basics = pd.read_csv(
    basics_path,
    sep="\t",
    na_values="\\N",
    low_memory=False
)

akas = pd.read_csv(
    akas_path,
    sep="\t",
    na_values="\\N",
    low_memory=False
)

basics["startYear"] = pd.to_numeric(basics["startYear"], errors="coerce")

In [64]:
movies = basics[
    (basics["titleType"] == "movie") &
    (basics["startYear"].between(2004, 2024)) &
    (basics["isAdult"] == 0)
].copy()

us_titles = akas[akas["region"] == "US"].copy()

us_movies = movies.merge(
    us_titles[["titleId", "title", "region"]],
    left_on="tconst",
    right_on="titleId",
    how="inner"
)
us_movies = us_movies.drop(columns="genres", errors="ignore")

In [72]:
import re

kaggle_df = pd.read_csv(
    DATA_RAW / "box_office_data_filtering(2000-2024).csv"
)

kaggle_df = kaggle_df.rename(columns={
    "Release Group": "kaggle_title",
    "Year": "kaggle_year",
    "Genres": "genres"
})

def normalize_title(title):
    if pd.isna(title):
        return None
    title = str(title).lower().strip()
    title = re.sub(r"[^a-z0-9\s]", "", title)
    title = re.sub(r"\s+", " ", title)
    return title

In [75]:
us_movies["normalized_title"] = us_movies["primaryTitle"].apply(normalize_title)
kaggle_df["normalized_title"] = kaggle_df["kaggle_title"].apply(normalize_title)

us_movies["startYear"] = pd.to_numeric(us_movies["startYear"], errors="coerce")
kaggle_df["kaggle_year"] = pd.to_numeric(kaggle_df["kaggle_year"], errors="coerce")

theatrical_movies = us_movies.merge(
    kaggle_df[["normalized_title", "kaggle_year","genres"]],
    left_on=["normalized_title", "startYear"],
    right_on=["normalized_title", "kaggle_year"],
    how="inner"
)

theatrical_movies = theatrical_movies.drop_duplicates(
    subset="tconst"
).reset_index(drop=True)

print(theatrical_movies.shape)

theatrical_movies[[
    "tconst",
    "primaryTitle",
    "startYear",
    "genres"
]].head()

(3881, 14)


,tconst,primaryTitle,startYear,genres
0,tt0120667,Fantastic Four,2005.0,"Action, Fantasy, Science Fiction"
1,tt0121164,Corpse Bride,2005.0,"Romance, Fantasy, Animation"
2,tt0121766,Star Wars: Episode III - Revenge of the Sith,2005.0,"Adventure, Action, Science Fiction"
3,tt0167190,Hellboy,2004.0,"Fantasy, Action"
4,tt0167456,Thunderbirds,2004.0,"Action, Adventure, Comedy, Family"


In [76]:
def safe_filename(text):
    return "".join(
        c if c.isalnum() else "_"
        for c in str(text).lower()
    )[:80]


def title_similarity(a, b):
    if pd.isna(a) or pd.isna(b):
        return 0

    return SequenceMatcher(
        None,
        str(a).lower(),
        str(b).lower()
    ).ratio()

In [77]:
def search_nyt_articles(title, year, max_pages=1, sleep_seconds=6):
    cache_path = DATA_CACHE / f"{safe_filename(title)}_{int(year)}.json"

    if cache_path.exists():
        with open(cache_path, "r", encoding="utf-8") as f:
            return json.load(f)

    all_docs = []

    for page in range(max_pages):
        params = {
            "q": title,
            "begin_date": f"{int(year)}0101",
            "end_date": f"{int(year)}1231",
            "sort": "relevance",
            "page": page,
            "api-key": NYT_API_KEY
        }

        response = requests.get(
            NYT_ARTICLE_SEARCH_URL,
            params=params,
            timeout=20
        )

        safe_url = response.url.replace(NYT_API_KEY, "[HIDDEN_KEY]")

        if response.status_code == 429:
            print("NYT rate limit hit. Stop and wait before continuing.")
            raise RuntimeError("NYT API rate limit hit")

        if response.status_code != 200:
            print(f"NYT status {response.status_code} for {title}: {safe_url}")
            print(response.text[:500])
            break

        docs = response.json().get("response", {}).get("docs") or []
        all_docs.extend(docs)

        if not docs:
            break

        time.sleep(sleep_seconds)

    with open(cache_path, "w", encoding="utf-8") as f:
        json.dump(all_docs, f, ensure_ascii=False, indent=2)

    return all_docs

In [78]:
def extract_nyt_review_features(title, year):
    docs = search_nyt_articles(title, year)

    movie_review_docs = [
        doc for doc in docs
        if doc.get("section_name") == "Movies"
        and doc.get("type_of_material") == "Review"
    ]

    if not movie_review_docs:
        return {
            "has_nyt_coverage": len(docs) > 0,
            "has_nyt_movie_review": False,
            "nyt_review_count": 0,
            "nyt_headline": None,
            "nyt_publication_date": None,
            "nyt_section": None,
            "nyt_material_type": None,
            "nyt_url": None
        }

    best_doc = movie_review_docs[0]

    headline = best_doc.get("headline", {}).get("main")
    pub_date = best_doc.get("pub_date")

    return {
        "has_nyt_coverage": len(docs) > 0,
        "has_nyt_movie_review": True,
        "nyt_review_count": len(movie_review_docs),
        "nyt_headline": headline,
        "nyt_publication_date": pub_date[:10] if pub_date else None,
        "nyt_section": best_doc.get("section_name"),
        "nyt_material_type": best_doc.get("type_of_material"),
        "nyt_url": best_doc.get("web_url")
    }

In [79]:
sample_movies = theatrical_movies.sample(
    min(SAMPLE_SIZE, len(us_movies)),
    random_state=42
).copy()

sample_movies = sample_movies[[
    "tconst",
    "primaryTitle",
    "startYear",
    "genres",
]].drop_duplicates()

sample_movies.head()

,tconst,primaryTitle,startYear,genres
3431,tt5998744,S Storm,2016.0,"Crime, Action, Thriller"
1018,tt0979434,Lottery Ticket,2010.0,Comedy
3270,tt4916630,Just Mercy,2019.0,"Drama, Crime, History"
3613,tt7390646,Black and Blue,2019.0,"Action, Crime, Drama, Thriller"
1200,tt10701074,Ponniyin Selvan: Part I,2022.0,"Action, Adventure"


In [80]:
rows = []

for _, movie in sample_movies.iterrows():
    title = movie["primaryTitle"]
    year = movie["startYear"]

    print(f"Searching NYT for: {title} ({int(year)})")

    row = movie.to_dict()

    try:
        review_features = extract_nyt_review_features(title, year)
        row.update(review_features)
    except RuntimeError:
        break

    rows.append(row)

nyt_sample_df = pd.DataFrame(rows)

nyt_sample_df.to_csv(
    DATA_PROCESSED / "imdb_nyt_review_sample.csv",
    index=False
)

print(nyt_sample_df.head(10))

Searching NYT for: S Storm (2016)
Searching NYT for: Lottery Ticket (2010)
Searching NYT for: Just Mercy (2019)
Searching NYT for: Black and Blue (2019)
Searching NYT for: Ponniyin Selvan: Part I (2022)
Searching NYT for: A Witness Out of the Blue (2019)
Searching NYT for: Love and Honor (2006)
NYT rate limit hit. Stop and wait before continuing.
       tconst               primaryTitle  startYear  \
0   tt5998744                    S Storm     2016.0   
1   tt0979434             Lottery Ticket     2010.0   
2   tt4916630                 Just Mercy     2019.0   
3   tt7390646             Black and Blue     2019.0   
4  tt10701074    Ponniyin Selvan: Part I     2022.0   
5  tt11076480  A Witness Out of the Blue     2019.0   

                           genres  has_nyt_coverage  has_nyt_movie_review  \
0         Crime, Action, Thriller              True                 False   
1                          Comedy              True                  True   
2           Drama, Crime, History 

In [36]:
display(nyt_sample_df.head(25))

,tconst,primaryTitle,startYear,genres,has_us_imdb_title,has_nyt_coverage,has_nyt_movie_review,nyt_review_count,nyt_headline,nyt_publication_date,nyt_section,nyt_material_type,nyt_url
0,tt0375735,Enduring Love,2004.0,"Drama,Mystery,Romance",True,True,False,0,NaN,NaN,NaN,NaN,NaN
1,tt0430683,Today,2004.0,NaN,True,True,False,0,NaN,NaN,NaN,NaN,NaN
2,tt0398638,30 Miles,2004.0,"Drama,Thriller",True,True,False,0,NaN,NaN,NaN,NaN,NaN
3,tt0381681,Before Sunset,2004.0,"Drama,Romance",True,True,True,2,FILM IN REVIEW; 'Mayor of the Sunset Strip',2004-04-02,Movies,Review,https://www.nytimes.com/2004/04/02/movies/film...
4,tt0382691,Gilles' Wife,2004.0,Drama,True,True,False,0,NaN,NaN,NaN,NaN,NaN
5,tt0814004,The Altruist,2004.0,"Crime,Thriller",True,True,False,0,NaN,NaN,NaN,NaN,NaN
6,tt37749008,Altered Courses,2004.0,Drama,True,True,False,0,NaN,NaN,NaN,NaN,NaN
7,tt0392049,The Good Cop,2004.0,"Action,Comedy",True,True,False,0,NaN,NaN,NaN,NaN,NaN
8,tt2981652,Mike and the Mechanics: Live at Shepherds Bush...,2004.0,Music,True,True,False,0,NaN,NaN,NaN,NaN,NaN
9,tt0296572,The Chronicles of Riddick,2004.0,"Action,Adventure,Sci-Fi",True,True,True,1,FILM REVIEW; Signs of Testosterone Are All Ove...,2004-06-11,Movies,Review,https://www.nytimes.com/2004/06/11/movies/film...


In [30]:
def debug_nyt_article_search(title, year, max_docs=10):
    params = {
        "q": title,
        "begin_date": f"{int(year)}0101",
        "end_date": f"{int(year)}1231",
        "sort": "relevance",
        "api-key": NYT_API_KEY
    }

    response = requests.get(
        NYT_ARTICLE_SEARCH_URL,
        params=params,
        timeout=20
    )

    safe_url = response.url.replace(NYT_API_KEY, "[HIDDEN_KEY]")

    print("STATUS:", response.status_code)
    print("URL:", safe_url)

    if response.status_code != 200:
        print(response.text[:1000])
        return []

    data = response.json()

    docs = data.get("response", {}).get("docs") or []

    print("DOC COUNT:", len(docs))
    print()

    for i, doc in enumerate(docs[:max_docs], start=1):
        print("=" * 80)
        print(f"RESULT {i}")
        print("HEADLINE:", doc.get("headline", {}).get("main"))
        print("SECTION:", doc.get("section_name"))
        print("TYPE:", doc.get("type_of_material"))
        print("DATE:", doc.get("pub_date"))
        print("URL:", doc.get("web_url"))

        abstract = doc.get("abstract")
        if abstract:
            print("ABSTRACT:", abstract)

        print()

    return docs

In [32]:
docs = debug_nyt_article_search("The Incredibles", 2004)

STATUS: 200
URL: https://api.nytimes.com/svc/search/v2/articlesearch.json?q=The+Incredibles&begin_date=20040101&end_date=20041231&sort=relevance&api-key=[HIDDEN_KEY]
DOC COUNT: 10

RESULT 1
HEADLINE: Arts, Briefly; The Incredible 'Incredibles'
SECTION: Arts
TYPE: News
DATE: 2004-11-15T05:00:00Z
URL: https://www.nytimes.com/2004/11/15/arts/movies/arts-briefly-the-incredible-incredibles.html
ABSTRACT: Arts, Briefly column; Disney-Pixar film The Incredibles dominates box office for second weekend in row, with estimated $51 million in box office sales; Warner Brothers' The Polar Express has disappointing first weekend, taking in $23.5 million; photo (S)

RESULT 2
HEADLINE: Disney and Pixar Score Again as 'The Incredibles' Opens Big
SECTION: Arts
TYPE: News
DATE: 2004-11-08T05:00:00Z
URL: https://www.nytimes.com/2004/11/08/movies/MoviesFeatures/disney-and-pixar-score-again-as-the-incredibles-opens.html
ABSTRACT: Pixar-Walt Disney Co computer-animated movie The Incredibles takes in estimated